# Tabel Variabel Data Pengangguran

Notebook ini merender tiga tabel langsung sebagai DataFrame di output Jupyter agar sel tabel dapat dipilih dan disalin. Tidak ada gambar atau file tabel yang disimpan.

Format mengikuti acuan: `No`, `Nama Variabel`, `Deskripsi`, `Skala`. Skala kategori mengikuti contoh (`Nominal`); deskripsi menjelaskan hitungan sesuai kolom sumber.

In [1]:
import pandas as pd
from IPython.display import display

from laporan_kerja_praktik import load_dataset

data = load_dataset()

EDUCATION_FIELDS = [
    "D1/D2/D3", "D4/S1", "MA", "MAK", "MI", "MTs",
    "Paket A", "Paket B", "Paket C", "Profesi", "S2", "S3",
    "SD", "SDLB", "SMA", "SMK", "SMLB", "SMP", "SMP LB",
    "SPM/PDF Ula", "SPM/PDF Ulya", "SPM/PDF Wustha",
    "Tidak Punya Ijazah SD",
]
EDUCATION_DESCRIPTIONS = {
    "D1/D2/D3": "Jumlah penganggur dengan pendidikan terakhir Diploma I, II, atau III.",
    "D4/S1": "Jumlah penganggur dengan pendidikan terakhir Diploma IV atau Sarjana (S1).",
    "MA": "Jumlah penganggur dengan pendidikan terakhir Madrasah Aliyah.",
    "MAK": "Jumlah penganggur dengan pendidikan terakhir Madrasah Aliyah Kejuruan.",
    "MI": "Jumlah penganggur dengan pendidikan terakhir Madrasah Ibtidaiyah.",
    "MTs": "Jumlah penganggur dengan pendidikan terakhir Madrasah Tsanawiyah.",
    "Paket A": "Jumlah penganggur pada kategori pendidikan terakhir Paket A.",
    "Paket B": "Jumlah penganggur pada kategori pendidikan terakhir Paket B.",
    "Paket C": "Jumlah penganggur pada kategori pendidikan terakhir Paket C.",
    "Profesi": "Jumlah penganggur pada kategori pendidikan terakhir profesi.",
    "S2": "Jumlah penganggur dengan pendidikan terakhir Magister (S2).",
    "S3": "Jumlah penganggur dengan pendidikan terakhir Doktor (S3).",
    "SD": "Jumlah penganggur dengan pendidikan terakhir Sekolah Dasar.",
    "SDLB": "Jumlah penganggur dengan pendidikan terakhir Sekolah Dasar Luar Biasa.",
    "SMA": "Jumlah penganggur dengan pendidikan terakhir Sekolah Menengah Atas.",
    "SMK": "Jumlah penganggur dengan pendidikan terakhir Sekolah Menengah Kejuruan.",
    "SMLB": "Jumlah penganggur dengan pendidikan terakhir Sekolah Menengah Luar Biasa.",
    "SMP": "Jumlah penganggur dengan pendidikan terakhir Sekolah Menengah Pertama.",
    "SMP LB": "Jumlah penganggur pada kategori pendidikan terakhir SMP Luar Biasa.",
    "SPM/PDF Ula": "Jumlah penganggur pada kategori SPM/PDF Ula sesuai label sumber.",
    "SPM/PDF Ulya": "Jumlah penganggur pada kategori SPM/PDF Ulya sesuai label sumber.",
    "SPM/PDF Wustha": "Jumlah penganggur pada kategori SPM/PDF Wustha sesuai label sumber.",
    "Tidak Punya Ijazah SD": "Jumlah penganggur yang tercatat tidak memiliki ijazah Sekolah Dasar.",
}

DISEASE_FIELDS = [
    "Tidak Ada", "Asma", "Diabetes", "Hipertensi", "Kanker/Tumor Ganas",
    "Kolesterol", "Gagal Ginjal", "Leukimia", "Masalah Jantung",
    "Rematik", "Stroke", "TBC", "Thalasemia", "Sirosis Hati",
    "HIV/AIDS", "Alzheimer", "Haemophilia", "Lainnya",
]
DISEASE_DESCRIPTIONS = {
    field: (
        "Jumlah penganggur yang tercatat tanpa penyakit kronis."
        if field == "Tidak Ada"
        else (
            "Jumlah penganggur pada kategori penyakit kronis lainnya sesuai label sumber."
            if field == "Lainnya"
            else f"Jumlah penganggur pada kategori penyakit kronis {field} sesuai label sumber."
        )
    )
    for field in DISEASE_FIELDS
}

SECTOR_FIELDS = [
    "Bangunan/Konstruksi", "Hotel & Rumah Makan", "Industri Pengolahan",
    "Informasi & Komunikasi",
    "Jasa Kemasyarakatan, Pemerintah, Dan Perorangan", "Jasa Kesehatan",
    "Jasa Pendidikan", "Keuangan & Asuransi", "Lainnya_2", "Listrik Dan Gas",
    "Perdagangan", "Perikanan", "Pertanian Padi & Palawija",
    "Peternakan", "Transportasi & Pergudangan",
]
SECTOR_DESCRIPTIONS = {
    field: f"Jumlah penduduk bekerja pada lapangan pekerjaan {field}."
    for field in SECTOR_FIELDS
}
SECTOR_DESCRIPTIONS["Lainnya_2"] = (
    "Jumlah penduduk bekerja pada kategori lapangan pekerjaan lainnya (nama kolom sumber: Lainnya_2)."
)

## Validasi kolom dan total sumber

Notebook memeriksa keberadaan kategori dan kecocokan total sebelum merender tabel. Kolom `Total` adalah ringkasan penyakit, bukan kategori. Header `#N/A` dikecualikan karena maknanya belum diklarifikasi; kategori `Tidak Ada` tetap dicantumkan.

In [2]:
category_fields = EDUCATION_FIELDS + DISEASE_FIELDS + SECTOR_FIELDS
required_fields = category_fields + ["Total_Penganggur", "Total_Penduduk_Bekerja", "Total"]
missing_fields = [field for field in required_fields if field not in data.columns]
if missing_fields:
    raise ValueError(f"Kolom wajib tidak ditemukan di CSV: {missing_fields}")

disease_detail_fields = [field for field in DISEASE_FIELDS if field != "Tidak Ada"]
checks = {
    "Kategori pendidikan = Total_Penganggur": data[EDUCATION_FIELDS]
    .sum(axis=1).eq(data["Total_Penganggur"]).all(),
    "Kategori sektor = Total_Penduduk_Bekerja": data[SECTOR_FIELDS]
    .sum(axis=1).eq(data["Total_Penduduk_Bekerja"]).all(),
    "Kategori penyakit = Total sumber": data[disease_detail_fields]
    .sum(axis=1).eq(data["Total"]).all(),
}
validasi = pd.DataFrame(
    {"Pemeriksaan": list(checks), "Lulus": list(checks.values())}
)
display(validasi.style.hide(axis="index"))
if not all(checks.values()):
    raise ValueError("Total kategori tidak cocok dengan total pada CSV sumber.")

def buat_tabel(fields, descriptions):
    return pd.DataFrame(
        {
            "No": range(1, len(fields) + 1),
            "Nama Variabel": fields,
            "Deskripsi": [descriptions[field] for field in fields],
            "Skala": ["Nominal"] * len(fields),
        }
    )

Pemeriksaan,Lulus
Kategori pendidikan = Total_Penganggur,True
Kategori sektor = Total_Penduduk_Bekerja,True
Kategori penyakit = Total sumber,True


## Tabel 3.2 — Pendidikan Terakhir

In [3]:
tabel_pendidikan = buat_tabel(EDUCATION_FIELDS, EDUCATION_DESCRIPTIONS)
display(tabel_pendidikan.style.hide(axis="index"))

No,Nama Variabel,Deskripsi,Skala
1,D1/D2/D3,"Jumlah penganggur dengan pendidikan terakhir Diploma I, II, atau III.",Nominal
2,D4/S1,Jumlah penganggur dengan pendidikan terakhir Diploma IV atau Sarjana (S1).,Nominal
3,MA,Jumlah penganggur dengan pendidikan terakhir Madrasah Aliyah.,Nominal
4,MAK,Jumlah penganggur dengan pendidikan terakhir Madrasah Aliyah Kejuruan.,Nominal
5,MI,Jumlah penganggur dengan pendidikan terakhir Madrasah Ibtidaiyah.,Nominal
6,MTs,Jumlah penganggur dengan pendidikan terakhir Madrasah Tsanawiyah.,Nominal
7,Paket A,Jumlah penganggur pada kategori pendidikan terakhir Paket A.,Nominal
8,Paket B,Jumlah penganggur pada kategori pendidikan terakhir Paket B.,Nominal
9,Paket C,Jumlah penganggur pada kategori pendidikan terakhir Paket C.,Nominal
10,Profesi,Jumlah penganggur pada kategori pendidikan terakhir profesi.,Nominal


## Tabel 3.3 — Penyakit Kronis

Kolom `Total` tidak ditampilkan karena merupakan ringkasan; header `#N/A` dikecualikan karena maknanya belum diklarifikasi. Kategori penyakit mengikuti label sumber dan tidak diasumsikan saling eksklusif.

In [4]:
tabel_penyakit = buat_tabel(DISEASE_FIELDS, DISEASE_DESCRIPTIONS)
display(tabel_penyakit.style.hide(axis="index"))

No,Nama Variabel,Deskripsi,Skala
1,Tidak Ada,Jumlah penganggur yang tercatat tanpa penyakit kronis.,Nominal
2,Asma,Jumlah penganggur pada kategori penyakit kronis Asma sesuai label sumber.,Nominal
3,Diabetes,Jumlah penganggur pada kategori penyakit kronis Diabetes sesuai label sumber.,Nominal
4,Hipertensi,Jumlah penganggur pada kategori penyakit kronis Hipertensi sesuai label sumber.,Nominal
5,Kanker/Tumor Ganas,Jumlah penganggur pada kategori penyakit kronis Kanker/Tumor Ganas sesuai label sumber.,Nominal
6,Kolesterol,Jumlah penganggur pada kategori penyakit kronis Kolesterol sesuai label sumber.,Nominal
7,Gagal Ginjal,Jumlah penganggur pada kategori penyakit kronis Gagal Ginjal sesuai label sumber.,Nominal
8,Leukimia,Jumlah penganggur pada kategori penyakit kronis Leukimia sesuai label sumber.,Nominal
9,Masalah Jantung,Jumlah penganggur pada kategori penyakit kronis Masalah Jantung sesuai label sumber.,Nominal
10,Rematik,Jumlah penganggur pada kategori penyakit kronis Rematik sesuai label sumber.,Nominal


## Tabel 3.4 — Sektor Pekerjaan

Nama kolom `Lainnya_2` dipertahankan sesuai sumber agar berbeda dari `Lainnya` pada kelompok penyakit.

In [5]:
tabel_sektor = buat_tabel(SECTOR_FIELDS, SECTOR_DESCRIPTIONS)
display(tabel_sektor.style.hide(axis="index"))

No,Nama Variabel,Deskripsi,Skala
1,Bangunan/Konstruksi,Jumlah penduduk bekerja pada lapangan pekerjaan Bangunan/Konstruksi.,Nominal
2,Hotel & Rumah Makan,Jumlah penduduk bekerja pada lapangan pekerjaan Hotel & Rumah Makan.,Nominal
3,Industri Pengolahan,Jumlah penduduk bekerja pada lapangan pekerjaan Industri Pengolahan.,Nominal
4,Informasi & Komunikasi,Jumlah penduduk bekerja pada lapangan pekerjaan Informasi & Komunikasi.,Nominal
5,"Jasa Kemasyarakatan, Pemerintah, Dan Perorangan","Jumlah penduduk bekerja pada lapangan pekerjaan Jasa Kemasyarakatan, Pemerintah, Dan Perorangan.",Nominal
6,Jasa Kesehatan,Jumlah penduduk bekerja pada lapangan pekerjaan Jasa Kesehatan.,Nominal
7,Jasa Pendidikan,Jumlah penduduk bekerja pada lapangan pekerjaan Jasa Pendidikan.,Nominal
8,Keuangan & Asuransi,Jumlah penduduk bekerja pada lapangan pekerjaan Keuangan & Asuransi.,Nominal
9,Lainnya_2,Jumlah penduduk bekerja pada kategori lapangan pekerjaan lainnya (nama kolom sumber: Lainnya_2).,Nominal
10,Listrik Dan Gas,Jumlah penduduk bekerja pada lapangan pekerjaan Listrik Dan Gas.,Nominal
